On test les Grad-CAM sur nos réseaux (https://arxiv.org/abs/1610.02391)

In [1]:
from retinotopy import *
from gradcam import *

Running on GPU :  NVIDIA GeForce RTX 3060 #GPU= 1
---------------------------------------------------------------------------------------------------
On date 2024-05-24, Running learning on host DESKTOP-27VNO0E with device cuda, pytorch==2.2.0+cu121
---------------------------------------------------------------------------------------------------
Welcome on Linux-5.10.16.3-microsoft-standard-WSL2-x86_64-with-glibc2.35
Random seed 1998 has been set.
Running on GPU :  NVIDIA GeForce RTX 3060 #GPU= 1


In [2]:
# The dataset to import images from

data_set_type = 'full'
args = Params()
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.folders = ['val'] # type of images to use
args.resolution = (7,7)
args.do_polar = False
image_datasets = image_datasets_transforms(args, shuffle=False, verbose=False)

In [3]:
for model_data_set_type in data_set_types:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(50*'.')

        print(f'{args.do_polar=}')

        print(50*'.')
        
        model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, args.do_polar) + '.pt'
        
        model = charge_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
        
        annotations = get_annotation('csv')

        df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, args.do_polar) + f'_complete_gradcam.parquet'
        print(df_filename)

        df_grad = None
        
        for i_image, (images, label) in tqdm(enumerate(image_datasets)):
        
            image_name = image_datasets.samples[i_image][0].split('/')[-1].split('.')[0]
            ground_true_indices, three_points, ground_true = get_ground_true(args, image_name, annotations)
            
            if ground_true.min() == 0.0:
                        
                images = images.to(device)
                
                since_grad = time.time()
                
                GradCam, pred = get_Grad_cam(model, images.unsqueeze(0), label)
    
                elapsed_time_grad = time.time() - since_grad
    
                arg_max_prior = torch.argmax(GradCam).item()
                #gradcam = np.array(GradCam.cpu().detach().numpy())
    
                GradCam = GradCam.reshape(args.resolution[0]*args.resolution[1])
    
                
                position_prior = (arg_max_prior%args.resolution[0], arg_max_prior//args.resolution[1])
    
                mid_point, in_point, ext_point = get_like_point(GradCam, args.resolution, three_points)
                
                grad_out = th_delete(GradCam, ground_true_indices[0])
        
                grad_out_max = torch.max(grad_out).item()
                grad_in_max = torch.max(GradCam[ground_true_indices[0]]).item()
            
                grad_out_mean = torch.mean(grad_out).item()
                grad_in_mean = torch.mean(GradCam[ground_true_indices[0]]).item()
    
                
                Iou = get_IoU(GradCam, ground_true.reshape(args.resolution[0]*args.resolution[1]))
    
                
    
                PG = 1 if grad_in_max > grad_out_max else 0
                
                
                df_grad_ = pd.DataFrame({'image_name':image_name, 'grad_in_max': grad_in_max, 'grad_out_max': grad_out_max,
                                         'grad_in_mean': grad_in_mean, 'grad_out_mean': grad_out_mean, 'position_prior':[position_prior],
                                         'Iou':[Iou], 'PG':PG,  'in_point':in_point.item(), 'mid_point':mid_point.item(), 'ext_point':ext_point.item(),
                                         'pred':pred, 'time_grad':elapsed_time_grad, 'label':label})
            
                df_grad = store_pandas(df_grad, df_grad_)
        
        df_grad.to_parquet(df_filename)
        model.cpu()

model_data_set_type='full'
..................................................
args.do_polar=False
..................................................
loading .... cached_data/2024-05-24_full_resnet18_cartesian.pt
cached_data/2024-05-24_full_resnet18_cartesian_complete_gradcam.parquet


97it [00:05, 18.47it/s]


KeyboardInterrupt: 

In [4]:
df_grad

,image_name,grad_in_max,grad_out_max,grad_in_mean,grad_out_mean,position_prior,Iou,PG,in_point,mid_point,ext_point,pred,time_grad,label
0,ILSVRC2012_val_00000293,1.0,0.914075,0.563409,0.240372,"(4, 4)","[0.234375, 0.30612244897959184, 0.318181818181...",1,0.914075,0.606429,0.014223,48,2.061483,0
1,ILSVRC2012_val_00002138,1.0,0.804810,0.712108,0.200088,"(1, 3)","[0.29508196721311475, 0.32727272727272727, 0.4...",1,1.000000,0.759628,0.067364,0,0.020435,0
2,ILSVRC2012_val_00003014,1.0,0.971752,1.000000,0.335746,"(3, 6)","[0.02040816326530612, 0.021739130434782608, 0....",1,1.000000,1.000000,0.050662,391,0.018311,0
3,ILSVRC2012_val_00006697,1.0,0.997795,0.652126,0.291432,"(3, 5)","[0.07547169811320754, 0.09302325581395349, 0.1...",1,1.000000,0.399790,0.031281,0,0.017072,0
4,ILSVRC2012_val_00007197,1.0,0.636554,0.625360,0.188694,"(3, 3)","[0.234375, 0.3191489361702128, 0.3589743589743...",1,0.887410,0.743955,0.002104,0,0.019650,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,ILSVRC2012_val_00046969,1.0,0.394099,0.611739,0.171514,"(2, 3)","[0.2898550724637681, 0.3333333333333333, 0.404...",1,0.379021,0.849589,0.019764,1,0.018437,1
93,ILSVRC2012_val_00047396,1.0,0.560826,0.669789,0.296154,"(2, 3)","[0.379746835443038, 0.38961038961038963, 0.422...",1,0.759865,0.754984,0.247458,1,0.019058,1
94,ILSVRC2012_val_00047561,1.0,0.521593,0.631132,0.184624,"(2, 3)","[0.234375, 0.3, 0.375, 0.36363636363636365, 0....",1,0.466325,1.000000,0.061019,1,0.019025,1
95,ILSVRC2012_val_00048840,1.0,0.770572,0.667872,0.207985,"(3, 3)","[0.19298245614035087, 0.2558139534883721, 0.30...",1,1.000000,0.488986,0.003097,1,0.017112,1
